# PhysIR — Stage 5: Ablation Suite @ 256 (Kaggle, ONE FLOW)

Runs **all 7 ablation configs back-to-back** — no SELECT switch. Idempotent + resume-safe:
trained configs are skipped, partial ones resume, so just **re-run the notebook** across
sessions/accounts until everything's done.

**Regime:** crop 256, eff-batch 8 (4×2-T4 DDP), phase 0.20 / retinex 0.18 / blur 0.05, EMA on,
60 ep — the sweep regime (rankings stable by ~30 ep). Footnote: extended ablations at 256px.

| config | losses | tests |
|--------|--------|-------|
| abl_baseline | pixel+edge+enhance | reference |
| abl_phase | +phase 0.20 | phase's effect |
| abl_retinex | +retinex 0.18 | retinex's effect |
| abl_full | +phase+retinex | combined |
| abl_blur | +blur 0.05 | blur standalone |
| abl_blurnoedge | +blur, edge OFF | blur as edge-substitute |
| abl_percep | +VGG perceptual | physics-vs-perceptual |

Each config → train, then eval on **LOLBlur test** (full-ref) + **per-image CSV** (for the
paired Wilcoxon test). **Accelerator: GPU T4 × 2.**

> ⚠️ Re-upload `physir-code` with the current `train.py` (resume fixes) first. To split across
> 2 accounts without switching: delete config lines from `CONFIGS` on one account.

## 1. Install

In [ ]:
!pip install -q lpips pytorch-msssim einops ptflops pyiqa

## 2. Paths (Kaggle input datasets)

In [ ]:
import os, sys, glob
CODE_DIR     = '/kaggle/input/datasets/jeffreyboahen/physir-code'
LOLBLUR_ROOT = '/kaggle/input/datasets/jeffreyboahen/lolblur-dataset'
WORKING_DIR = '/kaggle/working'
def find_root(base, marker, depth=3):
    if os.path.exists(os.path.join(base, marker)): return base
    for d in range(1, depth+1):
        h = sorted(glob.glob(os.path.join(base, *(['*']*d), marker)))
        if h: return h[0][:-len(marker)].rstrip(os.sep)
    return None
CODE_ROOT = find_root(CODE_DIR, 'train.py'); assert CODE_ROOT
if CODE_ROOT not in sys.path: sys.path.insert(0, CODE_ROOT)
src = open(os.path.join(CODE_ROOT,'train.py')).read()
miss = [n for n in ('restore_phys_state','class ModelEMA',"ckpt['best_psnr']",'class LPIPSLoss') if n not in src]
assert not miss, f'STALE physir-code, missing {miss} — re-upload.'
LOLBLUR = find_root(os.path.join(LOLBLUR_ROOT), os.path.join('train','high_sharp_scaled')) or LOLBLUR_ROOT
LOLBLUR = find_root(LOLBLUR, os.path.join('train','high_sharp_scaled')) or LOLBLUR
# resolve to the dir that has BOTH train/ and test/
cand = find_root(LOLBLUR_ROOT, os.path.join('test','low_blur_noise'))
LOLBLUR = cand or LOLBLUR
print('CODE_ROOT:', CODE_ROOT, '| fixes present ✅')
print('LOLBLUR  :', LOLBLUR, '| has test/:', os.path.isdir(os.path.join(LOLBLUR,'test')))

## 3. Train/val split (90/10, seed 42 — same as all stages)

In [ ]:
import random
from pathlib import Path
SUBS=['low_blur_noise','high_sharp_scaled']
TRAIN_SRC=Path(LOLBLUR)/'train'; SPLIT=Path(WORKING_DIR)/'LOLBlur_split'
random.seed(42)
clips=sorted([d.name for d in (TRAIN_SRC/'low_blur_noise').iterdir() if d.is_dir()])
n_val=max(1,int(len(clips)*0.1)); val=set(random.sample(clips,n_val)); tr=[c for c in clips if c not in val]
for nm,cl in [('physir_train',tr),('physir_val',val)]:
    for sub in SUBS:
        b=SPLIT/nm/sub; b.mkdir(parents=True,exist_ok=True)
        for c in cl:
            d=b/c
            if not d.exists(): d.symlink_to((TRAIN_SRC/sub/c).resolve())
PHYSIR_TRAIN=str(SPLIT/'physir_train'); PHYSIR_VAL=str(SPLIT/'physir_val')
print('train',len(tr),'val',len(val))

## 4. Ablation configs + shared config builder

In [ ]:
import yaml
EPOCHS = 60          # sweep regime; rankings stable by ~30 ep
PERCEP_W = 0.01      # DarkIR's lambda_pe (Eq.9). abl_percep uses LPIPS(VGG19) = DarkIR's L_percep

# each: (edge_on, phase_w, retinex_w, blur_w, percep_on)
# TIER 2 ONLY (for now). Tier 1 (baseline/+phase/+retinex/full) runs on Colab@384.
# NOTE: Tier-2 deltas + Wilcoxon need a 256 reference (baseline, and full for blur-noedge /
# physics-vs-perceptual). Uncomment 'abl_baseline' / 'abl_full' below to compute them in-flow.
CONFIGS = {
  # --- Tier 1 (Colab@384; uncomment only if you also want a 256 reference here) ---
  # 'abl_baseline':   dict(edge=True,  phase=0.0,  retinex=0.0,  blur=0.0,  percep=False),
  # 'abl_full':       dict(edge=True,  phase=0.20, retinex=0.18, blur=0.0,  percep=False),
  # --- Tier 2 (this run) — DarkIR-loss-equivalent (percep) FIRST, then blurs ---
  'abl_percep':     dict(edge=True,  phase=0.0,  retinex=0.0,  blur=0.0,  percep=True),
  'abl_blur':       dict(edge=True,  phase=0.0,  retinex=0.0,  blur=0.05, percep=False),
  'abl_blurnoedge': dict(edge=False, phase=0.0,  retinex=0.0,  blur=0.05, percep=False),
}

def build_cfg(name, sp):
    phys_on = (sp['phase']>0 or sp['retinex']>0 or sp['blur']>0)
    cfg = {
      'datasets': {'name':'LOLBlur',
        'train': {'low_path':f'{PHYSIR_TRAIN}/low_blur_noise','high_path':f'{PHYSIR_TRAIN}/high_sharp_scaled',
                  'crop_size':256,'batch_size':4,'n_workers':4,'use_flips':True},
        'val': {'test_path':PHYSIR_VAL,'batch_size_test':1,'verbose':False,'n_workers':4}},
      'network': {'name':'DarkIR','img_channels':3,'width':32,'middle_blk_num_enc':2,'middle_blk_num_dec':2,
                  'enc_blk_nums':[1,2,3],'dec_blk_nums':[3,1,1],'dilations':[1,4,9],'extra_depth_wise':True},
      'save': {'pretrained':None,'path':f'{WORKING_DIR}/{name}_latest.pt','best':f'{WORKING_DIR}/{name}_best.pt',
               'save_freq':5,'eval_freq':10,'perceptual_eval_epochs':[]},
      'pixel_criterion':'l1',
      'perceptual': sp['percep'], 'perceptual_weight':PERCEP_W,
      'perceptual_type':'lpips','perceptual_net':'vgg',   # faithful DarkIR L_percep = LPIPS(VGG19)
      'perceptual_criterion':'l1','perceptual_reduction':'mean',
      'edge': sp['edge'],'edge_weight':0.05,'edge_criterion':'l2','edge_reduction':'mean',
      'frequency':False,
      'enhance':True,'enhance_weight':0.01,'enhance_criterion':'l1','enhance_reduction':'mean',
      'physics_losses': {'enabled':phys_on,'phase2_start':9999,'phase3_start':9999,'phase4_start':1,
                         'retinex': sp['retinex']>0,'retinex_weight':sp['retinex'],
                         'blur': sp['blur']>0,'blur_weight':sp['blur'],
                         'phase': sp['phase']>0,'phase_weight':sp['phase']},
      'train': {'epochs':EPOCHS,'lr_initial':0.00028,'weight_decay':0.001,'lr_scheme':'CosineAnnealing',
                'eta_min':1e-6,'grad_clip':1.0,'ema':{'enabled':True,'decay':0.9999,'warmup':2000}},
      'wandb':{'init':False,'name':name},'Resize':False,
    }
    p=f'{WORKING_DIR}/{name}.yml'
    yaml.dump(cfg, open(p,'w'), default_flow_style=False)
    return p, cfg
print('configs:', list(CONFIGS))

## 5. TRAIN loop — all configs, idempotent + resume-safe

In [ ]:
import csv, subprocess
def done_ep(name):
    lg=Path(WORKING_DIR)/'.metrics'/name/'training_log.csv'
    if not lg.exists(): return 0
    last=0
    try:
        for r in csv.DictReader(open(lg)): last=max(last,int(r['epoch']))
    except Exception: pass
    return last

for name, sp in CONFIGS.items():
    d=done_ep(name)
    if d>=EPOCHS:
        print(f'✅ {name}: trained {d}/{EPOCHS} ep — skip'); continue
    p,_=build_cfg(name, sp)
    tag=f'(resume ep {d})' if d>0 else '(scratch)'
    print(f'\n===== TRAIN {name} {tag} =====')
    r=subprocess.run(['torchrun','--nproc_per_node=2',f'{CODE_ROOT}/train.py','-p',p])
    print(f'===== {name} done (exit {r.returncode}) =====')
print('\nTRAIN loop complete.')

## 6. EVAL loop — LOLBlur test (full-ref) + per-image CSV per config

In [ ]:
import torch, numpy as np, csv
import train as T
from data import create_test_data
device='cuda'
# LOLBlur test loader (full-res, shared across configs)
opt_test={'name':'LOLBlur','val':{'test_path':os.path.join(LOLBLUR,'test'),'batch_size_test':1},
          'train':{'n_workers':4,'verbose':True}}
tl_raw,_=create_test_data(rank=0,world_size=1,opt=opt_test)
tl = tl_raw if not isinstance(tl_raw,dict) else list(tl_raw.values())[0]['loader']
lpips_fn = T._LPIPS(net='vgg',verbose=False).to(device) if T._LPIPS_AVAILABLE else None
PERIMG=Path(WORKING_DIR)/'.metrics'/'per_image'; PERIMG.mkdir(parents=True,exist_ok=True)

net_cfg={'name':'DarkIR','img_channels':3,'width':32,'middle_blk_num_enc':2,'middle_blk_num_dec':2,
         'enc_blk_nums':[1,2,3],'dec_blk_nums':[3,1,1],'dilations':[1,4,9],'extra_depth_wise':True}

@torch.no_grad()
def eval_test(model, out_csv):
    rows=[]
    for i,(high,low) in enumerate(tl):
        high,low=high.to(device),low.to(device)
        pred=model(low,side_loss=False).clamp(0,1)
        _,_,H,W=high.shape; pred=pred[:,:,:H,:W]
        mse=torch.mean((pred-high)**2).item()
        psnr=20*np.log10(1.0/(np.sqrt(mse)+1e-8))
        T._ssim_metric.to(device); ssim=T._ssim_metric(pred,high).item()
        lp=lpips_fn(pred*2-1,high*2-1).mean().item()
        rows.append((i,psnr,ssim,lp))
    with open(out_csv,'w',newline='') as f:
        w=csv.writer(f); w.writerow(['idx','psnr','ssim','lpips']); w.writerows(rows)
    a=np.array([(p,s,l) for _,p,s,l in rows])
    return {'psnr':a[:,0].mean(),'ssim':a[:,1].mean(),'lpips':a[:,2].mean(),'n':len(a)}

summary={}
for name in CONFIGS:
    csvp=PERIMG/f'per_image_{name}.csv'
    best=f'{WORKING_DIR}/{name}_best.pt'
    if csvp.exists():
        print(f'✅ {name}: per-image exists — skip'); continue
    if not os.path.isfile(best):
        print(f'⏳ {name}: no checkpoint yet — train first'); continue
    ck=torch.load(best,map_location='cpu',weights_only=False)
    model=T.build_model(net_cfg).to(device)
    state=ck.get('ema') or ck['model']
    model.load_state_dict({k.replace('module.',''):v for k,v in state.items()},strict=False); model.eval()
    m=eval_test(model,csvp)
    summary[name]=m
    print(f'{name:16s} PSNR={m["psnr"]:.4f} SSIM={m["ssim"]:.4f} LPIPS={m["lpips"]:.4f}  -> {csvp.name}')
    del model; torch.cuda.empty_cache()
print('\nEVAL loop complete.')

## 7. Ablation leaderboard (LOLBlur test)

In [ ]:
import pandas as pd
rows=[]
for name in CONFIGS:
    csvp=Path(WORKING_DIR)/'.metrics'/'per_image'/f'per_image_{name}.csv'
    if not csvp.exists(): continue
    df=pd.read_csv(csvp)
    rows.append({'config':name,'psnr':df['psnr'].mean(),'ssim':df['ssim'].mean(),
                 'lpips':df['lpips'].mean(),'n':len(df)})
if rows:
    b=pd.DataFrame(rows).sort_values('psnr',ascending=False)
    print('=== Ablation @256 — LOLBlur test ===')
    print(b.to_string(index=False))
    if 'abl_baseline' in b['config'].values:
        base=b[b['config']=='abl_baseline'].iloc[0]
        print('\ndeltas vs baseline (PSNR):')
        for _,r in b.iterrows():
            print(f"  {r['config']:16s} {r['psnr']-base['psnr']:+.4f} dB")
    print('\nper-image CSVs in .metrics/per_image/ → feed paired_wilcoxon.py')
else:
    print('no results yet — run train + eval loops')